# 08 – Signals-only LSTM Baseline

Trains the signals-only LSTM baseline on the 16 hourly physiological variables plus their
observation masks (32 features x 24 hours).

**Run after notebooks 05 and 06** — uses hourly_vitals.csv and
modelling_cohort_sepsis_mortality.csv.

**Produces:** preds_signal.npz (test-set predictions, used by notebook 13).

MIMIC-III data not included (PhysioNet DUA); see README.


In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Load data

In [ ]:
vitals = pd.read_csv(data_path("hourly_vitals.csv"))
cohort = pd.read_csv(data_path("modelling_cohort_sepsis_mortality.csv"))

print("vitals:", vitals.shape)
print("cohort:", cohort.shape)

# label per ICU stay
labels = cohort[["ICUSTAY_ID", "mortality_after_24h"]].drop_duplicates()
print("positive rate:", labels["mortality_after_24h"].mean().round(3))

vitals: (805440, 34)
cohort: (10068, 14)
positive rate: 0.207


In [ ]:
VITAL_COLS = [
    "heart_rate",
    "sbp",
    "dbp",
    "map",
    "resp_rate",
    "temperature",
    "spo2",
    "fio2",
    "glucose",
    "ph",
    "gcs_eye",
    "gcs_motor",
    "gcs_total",
    "gcs_verbal",
    "weight",
    "height",
]

MASK_COLS = [c + "_observed" for c in VITAL_COLS]

# keep only stays that are in the modelling cohort, and sort so each stay's 24 hours are ordered
vitals = vitals[vitals["ICUSTAY_ID"].isin(labels["ICUSTAY_ID"])].copy()
vitals = vitals.sort_values(["ICUSTAY_ID","ICU_HOUR"]).reset_index(drop=True)

stay_ids = labels["ICUSTAY_ID"].values
y_all    = labels.set_index("ICUSTAY_ID")["mortality_after_24h"]
print("stays:", len(stay_ids))

stays: 10068


## 2. Patient-level split (before scaling)
Split on ICUSTAY_ID (one stay per patient in this cohort) so no patient appears in
more than one split. Stratify on the label to keep the positive rate similar across splits.

In [ ]:
train_ids, temp_ids = train_test_split(
    stay_ids, test_size=0.30, random_state=42,
    stratify=y_all.loc[stay_ids].values)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, random_state=42,
    stratify=y_all.loc[temp_ids].values)

print("train/val/test stays:", len(train_ids), len(val_ids), len(test_ids))
for name, ids in [("train",train_ids),("val",val_ids),("test",test_ids)]:
    print(f"  {name} positive rate:", round(float(y_all.loc[ids].mean()),3))

train/val/test stays: 7047 1510 1511
  train positive rate: 0.207
  val positive rate: 0.207
  test positive rate: 0.207


## 3. Standardise vitals using training-set statistics only

In [ ]:
train_rows = vitals[vitals["ICUSTAY_ID"].isin(train_ids)]
means = train_rows[VITAL_COLS].mean()
stds  = train_rows[VITAL_COLS].std().replace(0, 1.0)

vitals_scaled = vitals.copy()
vitals_scaled[VITAL_COLS] = (vitals_scaled[VITAL_COLS] - means) / stds
# residual missing (a vital never observed in 24h) -> 0 == training mean after scaling
vitals_scaled[VITAL_COLS] = vitals_scaled[VITAL_COLS].fillna(0.0)
print("standardised using training means/stds")

standardised using training means/stds


## 4. Build (n_stays, 24, 14) sequence tensors
Each stay becomes a 24-hour sequence with 14 features per hour: 7 scaled vitals + 7 masks.

In [ ]:
FEAT_COLS = VITAL_COLS + MASK_COLS  # 14 features

def build_tensor(ids):
    sub = vitals_scaled[vitals_scaled["ICUSTAY_ID"].isin(ids)]
    # group preserves 24 ordered rows per stay
    arr = (sub.set_index(["ICUSTAY_ID","ICU_HOUR"])[FEAT_COLS]
              .sort_index()
              .values
              .reshape(len(ids), 24, len(FEAT_COLS)))
    y = y_all.loc[ids].values.astype("float32")
    return torch.tensor(arr, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# NOTE: reshape assumes every stay has exactly 24 rows in ICU_HOUR order.
# Reindex to guarantee that ordering/shape:
def build_tensor_safe(ids):
    full_idx = pd.MultiIndex.from_product([ids, range(24)], names=["ICUSTAY_ID","ICU_HOUR"])
    sub = (vitals_scaled.set_index(["ICUSTAY_ID","ICU_HOUR"])
                        .reindex(full_idx)[FEAT_COLS]
                        .fillna(0.0))
    arr = sub.values.reshape(len(ids), 24, len(FEAT_COLS))
    y = y_all.loc[ids].values.astype("float32")
    return torch.tensor(arr, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

X_train, y_train = build_tensor_safe(train_ids)
X_val,   y_val   = build_tensor_safe(val_ids)
X_test,  y_test  = build_tensor_safe(test_ids)
print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

X_train: torch.Size([7047, 24, 32]) | X_val: torch.Size([1510, 24, 32]) | X_test: torch.Size([1511, 24, 32])


In [ ]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=256)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=256)

## 5. LSTM model

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, n_features=len(FEAT_COLS), hidden=64, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            n_features,
            hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, (h_n, _) = self.lstm(x)
        last = h_n[-1]
        return self.fc(self.dropout(last)).squeeze(1)

# fix random seed for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model = LSTMClassifier().to(device)
print(model)

LSTMClassifier(
  (lstm): LSTM(32, 64, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


## 6. Train with class weighting

In [ ]:
pos_rate = float(y_train.mean())
pos_weight = torch.tensor([(1 - pos_rate) / pos_rate], device=device)  # up-weight the minority class
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            ps.append(probs); ys.append(yb.numpy())
    y_true = np.concatenate(ys); y_prob = np.concatenate(ps)
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, (y_prob >= 0.5).astype(int))
    return auroc, auprc, f1, y_true, y_prob

best_val_auroc, best_state, patience, wait = 0, None, 5, 0
for epoch in range(1, 41):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    va_auroc, va_auprc, va_f1, *_ = evaluate(val_loader)
    print(f"epoch {epoch:2d} | val AUROC {va_auroc:.3f} | val AUPRC {va_auprc:.3f} | val F1 {va_f1:.3f}")
    if va_auroc > best_val_auroc:
        best_val_auroc, best_state, wait = va_auroc, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
    else:
        wait += 1
        if wait >= patience:
            print("early stopping"); break

model.load_state_dict(best_state)

epoch  1 | val AUROC 0.694 | val AUPRC 0.394 | val F1 0.431
epoch  2 | val AUROC 0.699 | val AUPRC 0.409 | val F1 0.421
epoch  3 | val AUROC 0.696 | val AUPRC 0.412 | val F1 0.411
epoch  4 | val AUROC 0.704 | val AUPRC 0.418 | val F1 0.438
epoch  5 | val AUROC 0.706 | val AUPRC 0.416 | val F1 0.439
epoch  6 | val AUROC 0.707 | val AUPRC 0.415 | val F1 0.440
epoch  7 | val AUROC 0.699 | val AUPRC 0.411 | val F1 0.430
epoch  8 | val AUROC 0.698 | val AUPRC 0.412 | val F1 0.425
epoch  9 | val AUROC 0.698 | val AUPRC 0.415 | val F1 0.427
epoch 10 | val AUROC 0.698 | val AUPRC 0.417 | val F1 0.429
epoch 11 | val AUROC 0.697 | val AUPRC 0.419 | val F1 0.424
early stopping


<All keys matched successfully>

## 7. Final test-set performance

In [ ]:
te_auroc, te_auprc, te_f1, y_true, y_prob = evaluate(test_loader)
print("=== Signal-only LSTM baseline (test set) ===")
print(f"AUROC: {te_auroc:.3f}")
print(f"AUPRC: {te_auprc:.3f}")
print(f"F1   : {te_f1:.3f}")
print(f"n_test: {len(y_true)} | positives: {int(y_true.sum())} ({y_true.mean():.1%})")

=== Signal-only LSTM baseline (test set) ===
AUROC: 0.675
AUPRC: 0.379
F1   : 0.413
n_test: 1511 | positives: 313 (20.7%)


In [ ]:
te_auroc, te_auprc, te_f1, y_true, y_prob = evaluate(test_loader)
np.savez(data_path("preds_signal.npz"), stay_ids=np.array(test_ids), y_true=y_true, y_prob=y_prob)
print("saved preds_signal.npz", y_true.shape)

saved preds_signal.npz (1511,)
